<a href="https://colab.research.google.com/github/Suhana15-S/GEN_AI_CA/blob/main/GEN_AI_CA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

In [ ]:
import pandas as pd
import random

base_data = [
    ("i am happy", "मैं खुश हूँ", "हम खुश बानी"),
    ("i am sad", "मैं उदास हूँ", "हम उदास बानी"),
    ("i love tea", "मुझे चाय पसंद है", "हम चाय पसंद करीं"),
    ("i love coffee", "मुझे कॉफी पसंद है", "हम कॉफी पसंद करीं"),
    ("i love machine learning", "मुझे मशीन लर्निंग पसंद है", "हम मशीन लर्निंग पसंद करीं"),
    ("he is going home", "वह घर जा रहा है", "ऊ घर जा रहल बा"),
    ("she is studying", "वह पढ़ रही है", "ऊ पढ़ रहल बिया"),
    ("we are friends", "हम दोस्त हैं", "हमनी दोस्त बानी"),
    ("they are playing", "वे खेल रहे हैं", "ऊ सब खेल रहल बाड़े"),
    ("this is good", "यह अच्छा है", "ई बढ़िया बा"),
    ("this is bad", "यह बुरा है", "ई खराब बा"),
    ("i am learning ai", "मैं एआई सीख रहा हूँ", "हम एआई सीख रहल बानी")
]

input_texts = []
output_texts = []

for _ in range(10000):
    row = random.choice(base_data)
    langs = ["english", "hindi", "magahi"]

    input_lang = random.choice(langs)
    output_lang = random.choice([l for l in langs if l != input_lang])

    if input_lang == "english": in_text = row[0]
    elif input_lang == "hindi": in_text = row[1]
    else: in_text = row[2]

    if output_lang == "english": out_text = row[0]
    elif output_lang == "hindi": out_text = row[1]
    else: out_text = row[2]

    in_text = f"<from_{input_lang}> {in_text}"
    out_text = f"<to_{output_lang}> {out_text}"

    input_texts.append(in_text)
    output_texts.append(out_text)

dataset = pd.DataFrame({
    "input_text": input_texts,
    "output_text": output_texts
})

# save to CSV
dataset.to_csv("3lang_dataset.csv", index=False)

dataset.head(10)

,input_text,output_text
0,<from_hindi> वह घर जा रहा है,<to_magahi> ऊ घर जा रहल बा
1,<from_magahi> हम उदास बानी,<to_english> i am sad
2,<from_english> we are friends,<to_magahi> हमनी दोस्त बानी
3,<from_hindi> मुझे कॉफी पसंद है,<to_english> i love coffee
4,<from_magahi> हम खुश बानी,<to_english> i am happy
5,<from_hindi> वह पढ़ रही है,<to_magahi> ऊ पढ़ रहल बिया
6,<from_english> she is studying,<to_magahi> ऊ पढ़ रहल बिया
7,<from_hindi> मैं एआई सीख रहा हूँ,<to_magahi> हम एआई सीख रहल बानी
8,<from_english> i am sad,<to_hindi> मैं उदास हूँ
9,<from_english> i love tea,<to_magahi> हम चाय पसंद करीं


In [ ]:
dataset.to_csv("multilingual_dataset.csv", index=False)

In [ ]:
dataset.shape


(10000, 2)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

input_tokenizer = Tokenizer(filters='', lower=True, oov_token="<OOV>")
input_tokenizer.fit_on_texts(dataset['input_text'])
encoder_input_data = input_tokenizer.texts_to_sequences(dataset['input_text'])

target_tokenizer = Tokenizer(filters='', lower=True, oov_token="<OOV>")
target_tokenizer.fit_on_texts(dataset['output_text'])
decoder_input_data = target_tokenizer.texts_to_sequences(dataset['output_text'])

max_encoder_len = max(len(seq) for seq in encoder_input_data)
max_decoder_len = max(len(seq) for seq in decoder_input_data)

encoder_input_data = pad_sequences(encoder_input_data, maxlen=max_encoder_len, padding='post')
decoder_input_data = pad_sequences(decoder_input_data, maxlen=max_decoder_len, padding='post')

decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

In [ ]:
from sklearn.model_selection import train_test_split

encoder_input_train, encoder_input_test, \
decoder_input_train, decoder_input_test, \
decoder_target_train, decoder_target_test = train_test_split(
    encoder_input_data,
    decoder_input_data,
    decoder_target_data,
    test_size=0.2,
    random_state=42
)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

latent_dim = 256
num_encoder_tokens = len(input_tokenizer.word_index) + 1
num_decoder_tokens = len(target_tokenizer.word_index) + 1

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(num_decoder_tokens, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, None, 256) │     17,664 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, None, 256) │     17,664 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_4[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, None,     │    525,312 │ embedding_5[0][0… │
│                     │ 256), (None,      │            │ lstm_4[0][1],     │
│                     │ 256), (None,      │            │ lstm_4[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 69)  │     17,733 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,103,685 (4.21 MB)

 Trainable params: 1,103,685 (4.21 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 64
epochs = 20

model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_target_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - accuracy: 0.4398 - loss: 2.7160 - val_accuracy: 0.9597 - val_loss: 0.2576
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - accuracy: 0.9784 - loss: 0.1332 - val_accuracy: 1.0000 - val_loss: 0.0150
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 114ms/step - accuracy: 1.0000 - loss: 0.0116 - val_accuracy: 1.0000 - val_loss: 0.0058
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 18s 93ms/step - accuracy: 1.0000 - loss: 0.0050 - val_accuracy: 1.0000 - val_loss: 0.0032
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 104ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 115ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 117ms/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 8/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 104ms/step - accuracy: 1.0000 - loss: 0.

In [ ]:
from tensorflow.keras.models import Model

encoder_model = Model(encoder_inputs, encoder_states)

In [ ]:
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [ ]:
def translate(input_text, input_lang, output_lang):
    input_text = f"<from_{input_lang}> {input_text}"
    seq = input_tokenizer.texts_to_sequences([input_text])
    seq = pad_sequences(seq, maxlen=encoder_input_train.shape[1], padding='post')

    states_value = encoder_model.predict(seq)

    target_seq = np.array([[target_tokenizer.word_index[f"<to_{output_lang}>"]]])
    decoded_sentence = ""
    stop_condition = False

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = {v:k for k,v in target_tokenizer.word_index.items()}.get(sampled_token_index, "")

        if sampled_word == "" or len(decoded_sentence.split()) > max_decoder_len:
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]

    return decoded_sentence.strip()

In [ ]:
import gradio as gr

def gui_translate(input_text, input_lang, output_lang):
    if input_text.strip() == "":
        return "Please enter some text."
    return translate(input_text, input_lang, output_lang)

interface = gr.Interface(
    fn=gui_translate,
    inputs=[
        gr.Textbox(label="Enter Text"),
        gr.Dropdown(["english", "hindi", "magahi"], label="Input Language"),
        gr.Dropdown(["english", "hindi", "magahi"], label="Output Language")
    ],
    outputs="text",
    title="3-Language Translator (LSTM)",
    description="Enter text, select input and output languages, click Translate."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0735e0a00d69b8702.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
